# Treino do DiffusionPen com acentos no Colab

Roda um treino do `iam_acentuado` numa GPU do Colab, com o mesmo comando da máquina local. A sessão do Colab pode cair. Por isso o estado do treino é copiado para o Google Drive ao fim de cada época, e rodar o notebook de novo retoma de onde parou.

Antes da primeira vez
1. Na máquina que tem os dados, rode `bash scripts/pacote_nuvem.sh`. Ele cria `pacote_nuvem.tar`, com cerca de 3 GB.
2. No Google Drive, crie a pasta `htg-tcc` e ponha o `pacote_nuvem.tar` dentro. Deixe a pasta privada, o IAM exige cadastro. O Drive precisa de uns 6 GB livres, 2,8 do pacote e 3 do treino.
3. Aqui no Colab, em Ambiente de execução, escolha uma GPU.

Depois é só preencher a célula de configuração e rodar as células em ordem. Numa sessão nova, rode tudo de novo com a mesma configuração.

## 1. Configuração

In [ ]:
NOME = "sem_pares"          # o treino fica em treinos/model_<NOME> no Drive
EPOCAS = 16                 # total; ao retomar, treina so o que falta
# O que muda em relacao ao treino com professor (peso 5 no acento, professor = modelo do IAM):
ARGS = "--acento_separado --professor_sem_pares"
#   zona vogais:            "--acento_separado --peso_zona 5 --zona vogais"
#   professor de referencia: "--acento_separado"

BRANCH = "diagnostico-gpu-e-reorganizacao"
REPO = "https://github.com/vitorNicoletti/HTG-Research-Pt-Br.git"
PASTA_DRIVE = "/content/drive/MyDrive/htg-tcc"
RAIZ = "/content/htg"

## 2. Placa e Drive

In [ ]:
import os, subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "sem GPU: escolha uma em Ambiente de execucao > Alterar o tipo"
print("torch", torch.__version__, "| nucleos de CPU:", os.cpu_count())

from google.colab import drive
drive.mount("/content/drive")
assert os.path.isfile(f"{PASTA_DRIVE}/pacote_nuvem.tar"), f"ponha o pacote_nuvem.tar em {PASTA_DRIVE}"
os.makedirs(f"{PASTA_DRIVE}/treinos", exist_ok=True)
os.makedirs(f"{PASTA_DRIVE}/resultados", exist_ok=True)

## 3. Código, dados e bibliotecas

Leva alguns minutos na primeira vez de cada sessão. O clone do DiffusionPen fica no mesmo commit usado nas outras máquinas.

In [ ]:
if not os.path.isdir(RAIZ):
    !git clone -q --branch {BRANCH} {REPO} {RAIZ}
%cd {RAIZ}
!git pull -q && git log -1 --format="repositorio em %h %s"
if not os.path.isdir("DiffusionPen/utils"):
    !git clone -q https://github.com/koninik/DiffusionPen.git DiffusionPen
    !git -C DiffusionPen checkout -q 059e2f2bcb7372231038719e955597917c1a8783
!bash scripts/aplicar_mods.sh | tail -1

if not os.path.isdir("DiffusionPen/iam_data/words"):
    !tar xf {PASTA_DRIVE}/pacote_nuvem.tar -C {RAIZ}
n = int(subprocess.run("find DiffusionPen/iam_data/words -name '*.png' | wc -l", shell=True, capture_output=True, text=True).stdout)
assert n >= 115000, f"IAM incompleto: {n} imagens"
print("imagens do IAM:", n)

!pip install -q einops timm diffusers transformers wandb scikit-image
os.environ["WANDB_MODE"] = "disabled"
import diffusers, transformers, timm
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__, "| timm", timm.__version__)

## 4. Testes da placa

Regra do projeto para toda placa nova. Comparam a conta da GPU com a da CPU.

- no primeiro, o erro deve ficar perto de 1e-6
- no segundo, nenhum gradiente não finito
- no terceiro, o cosseno deve dar perto de 1,0

Se algum falhar, não treine nesta sessão.

In [ ]:
os.environ["CKPT"] = "./DiffusionPen/diffusionpen_iam_model_path/models/ema_ckpt.pt"
!python diagnostico/teste_conv_isolada.py
!python diagnostico/teste_gradiente_sintetico.py
!python diagnostico/teste_direcao_gradiente.py

## 5. Teste de fumaça

Uma época com 64 amostras, só para ver o comando rodar do começo ao fim. Não vai para o Drive.

In [ ]:
os.environ.update(NOME="fumaca", EPOCAS="1", DESTINO="/content/fumaca", ARGS=ARGS + " --max_samples 64")
!rm -rf model_fumaca /content/fumaca && bash scripts/treino_nuvem.sh && rm -rf model_fumaca /content/fumaca

## 6. Treino

Mostra uma linha por época. A cópia para o Drive acontece sozinha depois de cada uma. Se a sessão cair, abra de novo, rode as células 1 a 3 e esta, e ele continua.

A linha `professor em N% das amostras` confere a opção escolhida. Com `--professor_sem_pares` deve dar perto de 70%, sem ela perto de 85%.

In [ ]:
os.environ.update(NOME=NOME, EPOCAS=str(EPOCAS), DESTINO=f"{PASTA_DRIVE}/treinos", ARGS=ARGS)
!bash scripts/treino_nuvem.sh

## 7. Avaliação

O protocolo de sempre, 3.200 imagens por modelo, nos pontos de 6 épocas e final. O resultado vai para `resultados/` no Drive, num arquivo só, pronto para baixar.

In [ ]:
m = f"model_{NOME}/models"
pontos = [(f"{NOME}_6ep", f"{m}/ema_ep5.pt"), (f"{NOME}_{EPOCAS}ep", f"{m}/ema_ckpt.pt")]
modelos = " ".join(f"--modelo {r}={c}" for r, c in pontos if os.path.isfile(c))
saida = f"avaliacao_{NOME}_val"
!python scripts/avaliar_pt.py --split val {modelos} --bases iam_pt_alinhado --leitor modelos/leitor_iam_transformer.pt --saida {saida}
!cat {saida}/resumo.md
!tar czf {PASTA_DRIVE}/resultados/{saida}.tgz {saida} model_{NOME}/config.jsonl model_{NOME}/treino.log
print("no Drive:", f"{PASTA_DRIVE}/resultados/{saida}.tgz")

## Depois

Baixe o `.tgz` de `resultados/` e ponha na raiz do repositório local, como os outros.

Cada treino ocupa perto de 2,3 GB no Drive, e a cópia precisa de 0,7 GB de folga. Quando a avaliação estiver feita, dá para apagar de `treinos/model_<NOME>/models` o `ckpt.pt` e o `optim.pt`, que só servem para retomar. O `ema_ckpt.pt` é o modelo. Arquivo apagado no Drive continua contando no espaço até a lixeira ser esvaziada.